# 05_03 Aggregation analysis for the selected global LightGBM

This notebook tests whether the high daily pooled WAPE for `Pseudo 890` mainly reflects irreducible article-store-day arrival noise or remaining model error. It reuses the two-stage daily forecasts selected in `05_01`; no model is retrained.

Pseudo 890 averages about 1.16 kg per active series-day. At that volume, many observations represent one or zero purchase events, so even a well-calibrated conditional-mean forecast can have large daily absolute error. Aggregating the same forecasts reveals how much of that error cancels at a reporting grain with more signal.

## Design

The same active Pseudo 890 forecast rows are evaluated at three grains:

1. `article-store-day`: the original forecast grain;
2. `article-store-week`: actuals and forecasts summed over each seven-day origin horizon per series;
3. `article-day across stores`: actuals and forecasts summed over stores for each article and target day.

Pooled WAPE is recomputed *after* aggregation. Actual and forecast kilograms must remain identical across all three grains. Therefore, WAPE reductions measure cancellation of signed row errors rather than a change in the evaluated population. The preregistered interpretation is: weekly WAPE around 25–35% indicates a largely noise-limited daily problem, while weekly WAPE near 55% indicates substantial modelling headroom.

In [1]:
from pathlib import Path
import sys

import duckdb
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'src').exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.models.benchmark import load_benchmark_design
from src.models.lightgbm import LIGHTGBM_MODEL_LABELS, TWO_STAGE_MODEL_NAME
from src.models.results import result_path

pd.set_option('display.max_columns', 40)
pd.set_option('display.max_rows', 60)

ANALYSIS_MODEL = TWO_STAGE_MODEL_NAME
SOURCE_GROUP = 'Pseudo'
CATEGORY_ID = 890
GRAIN_ORDER = [
    'article_store_day', 'article_store_week', 'article_day_across_stores'
]
GRAIN_LABELS = {
    'article_store_day': 'Article-store-day',
    'article_store_week': 'Article-store-week (7-day total)',
    'article_day_across_stores': 'Article-day (stores pooled)',
}

In [2]:
design = load_benchmark_design()
forecast_path = result_path(ANALYSIS_MODEL, design)

con = duckdb.connect()
con.execute('PRAGMA threads=4')
con.execute(
    '''
    CREATE OR REPLACE TEMP TABLE pseudo_890_daily AS
    SELECT
        ARTIKEL_ID::BIGINT AS ARTIKEL_ID,
        MARKT_ID::BIGINT AS MARKT_ID,
        CAST(origin AS DATE) AS origin,
        CAST(period AS DATE) AS period,
        actual::DOUBLE AS actual,
        forecast::DOUBLE AS forecast
    FROM read_csv_auto(?)
    WHERE is_active
      AND sourcing_group = ?
      AND category_id = ?
    ''',
    [str(forecast_path), SOURCE_GROUP, CATEGORY_ID],
)

data_audit = con.execute(
    '''
    SELECT
        COUNT(*) AS active_series_days,
        COUNT(DISTINCT (ARTIKEL_ID, MARKT_ID)) AS series,
        COUNT(DISTINCT ARTIKEL_ID) AS articles,
        COUNT(DISTINCT MARKT_ID) AS stores,
        COUNT(DISTINCT origin) AS origins,
        MIN(origin) AS first_origin,
        MAX(origin) AS last_origin,
        AVG(actual) AS mean_actual_kg_per_active_series_day,
        AVG((actual > 0)::INTEGER) AS occurrence_rate,
        AVG(actual) FILTER (WHERE actual > 0) AS positive_quantity_mean
    FROM pseudo_890_daily
    '''
).fetchdf()
if data_audit.loc[0, 'active_series_days'] == 0:
    raise ValueError('No active Pseudo 890 forecast rows were found')

print(f'Model: {LIGHTGBM_MODEL_LABELS[ANALYSIS_MODEL]}')
print(f'Forecast artifact: {forecast_path.relative_to(ROOT)}')
display(data_audit.style.format({
    'active_series_days': '{:,.0f}',
    'series': '{:,.0f}', 'articles': '{:,.0f}', 'stores': '{:,.0f}',
    'origins': '{:,.0f}', 'first_origin': '{:%Y-%m-%d}',
    'last_origin': '{:%Y-%m-%d}',
    'mean_actual_kg_per_active_series_day': '{:.3f}',
    'occurrence_rate': '{:.1%}',
    'positive_quantity_mean': '{:.3f}',
}))

Model: Two-stage LightGBM (occurrence × quantity)
Forecast artifact: reports/results/forecasts/artikel_markt_multi7days_lightgbm_two_stage.csv


,active_series_days,series,articles,stores,origins,first_origin,last_origin,mean_actual_kg_per_active_series_day,occurrence_rate,positive_quantity_mean
0,"2,142,197","18,619",278,193,20,2026-03-02,2026-07-13,1.162,37.8%,3.077


## Recomputed WAPE after aggregation

The two aggregated tables below are the only transformations: one groups over target days within each series-origin, and one groups over stores within each article-target-day.

In [3]:
con.execute(
    '''
    CREATE OR REPLACE TEMP TABLE pseudo_890_weekly AS
    SELECT
        ARTIKEL_ID, MARKT_ID, origin,
        SUM(actual) AS actual,
        SUM(forecast) AS forecast
    FROM pseudo_890_daily
    GROUP BY ARTIKEL_ID, MARKT_ID, origin
    '''
)
con.execute(
    '''
    CREATE OR REPLACE TEMP TABLE pseudo_890_article_day AS
    SELECT
        ARTIKEL_ID, origin, period,
        SUM(actual) AS actual,
        SUM(forecast) AS forecast
    FROM pseudo_890_daily
    GROUP BY ARTIKEL_ID, origin, period
    '''
)

grain_summary = con.execute(
    '''
    WITH grains AS (
        SELECT 'article_store_day' AS grain, actual, forecast
        FROM pseudo_890_daily
        UNION ALL
        SELECT 'article_store_week' AS grain, actual, forecast
        FROM pseudo_890_weekly
        UNION ALL
        SELECT 'article_day_across_stores' AS grain, actual, forecast
        FROM pseudo_890_article_day
    )
    SELECT
        grain,
        COUNT(*) AS n_units,
        SUM(actual) AS actual_kg,
        SUM(forecast) AS forecast_kg,
        AVG(actual) AS mean_actual_kg_per_unit,
        SUM(ABS(forecast - actual)) AS absolute_error_kg,
        SUM(ABS(forecast - actual)) / NULLIF(SUM(actual), 0) AS pooled_wape,
        SUM(forecast) / NULLIF(SUM(actual), 0) AS ratio
    FROM grains
    GROUP BY grain
    '''
).fetchdf()
grain_summary['grain'] = pd.Categorical(
    grain_summary.grain, categories=GRAIN_ORDER, ordered=True
)
grain_summary = grain_summary.sort_values('grain').reset_index(drop=True)

if not np.allclose(grain_summary.actual_kg, grain_summary.actual_kg.iloc[0]):
    raise ValueError('Actual kilograms changed across aggregation grains')
if not np.allclose(grain_summary.forecast_kg, grain_summary.forecast_kg.iloc[0]):
    raise ValueError('Forecast kilograms changed across aggregation grains')

daily_wape = grain_summary.loc[
    grain_summary.grain.eq('article_store_day'), 'pooled_wape'
].iloc[0]
daily_abs_error = grain_summary.loc[
    grain_summary.grain.eq('article_store_day'), 'absolute_error_kg'
].iloc[0]
grain_summary['wape_reduction_vs_daily_pp'] = 100 * (
    daily_wape - grain_summary.pooled_wape
)
grain_summary['absolute_error_removed_vs_daily'] = (
    1 - grain_summary.absolute_error_kg / daily_abs_error
)
grain_summary['grain_label'] = grain_summary.grain.map(GRAIN_LABELS)

display(grain_summary[[
    'grain_label', 'n_units', 'mean_actual_kg_per_unit',
    'actual_kg', 'forecast_kg', 'pooled_wape', 'ratio',
    'wape_reduction_vs_daily_pp', 'absolute_error_removed_vs_daily',
]].style.format({
    'n_units': '{:,.0f}',
    'mean_actual_kg_per_unit': '{:,.3f}',
    'actual_kg': '{:,.1f}', 'forecast_kg': '{:,.1f}',
    'pooled_wape': '{:.2%}', 'ratio': '{:.3f}',
    'wape_reduction_vs_daily_pp': '{:+.2f}',
    'absolute_error_removed_vs_daily': '{:.1%}',
}))

,grain_label,n_units,mean_actual_kg_per_unit,actual_kg,forecast_kg,pooled_wape,ratio,wape_reduction_vs_daily_pp,absolute_error_removed_vs_daily
0,Article-store-day,"2,142,197",1.162,"2,489,426.8","2,463,586.9",57.53%,0.990,+0.00,0.0%
1,Article-store-week (7-day total),"371,877",6.694,"2,489,426.8","2,463,586.9",37.37%,0.990,+20.16,35.0%
2,Article-day (stores pooled),"34,296",72.587,"2,489,426.8","2,463,586.9",25.72%,0.990,+31.81,55.3%


## Stability across evaluation origins

The pooled result can hide difficult weeks. Recomputing WAPE within each origin checks whether the aggregation effect is systematic rather than driven by a small number of high-volume origins.

In [4]:
origin_results = con.execute(
    '''
    WITH grains AS (
        SELECT 'article_store_day' AS grain, origin, actual, forecast
        FROM pseudo_890_daily
        UNION ALL
        SELECT 'article_store_week' AS grain, origin, actual, forecast
        FROM pseudo_890_weekly
        UNION ALL
        SELECT 'article_day_across_stores' AS grain, origin, actual, forecast
        FROM pseudo_890_article_day
    )
    SELECT
        grain, origin,
        SUM(ABS(forecast - actual)) / NULLIF(SUM(actual), 0) AS pooled_wape
    FROM grains
    GROUP BY grain, origin
    ORDER BY origin, grain
    '''
).fetchdf()
origin_results['grain_label'] = origin_results.grain.map(GRAIN_LABELS)
origin_stability = (
    origin_results.groupby(['grain', 'grain_label'], observed=True)
    .agg(
        origins=('origin', 'nunique'),
        minimum_origin_wape=('pooled_wape', 'min'),
        median_origin_wape=('pooled_wape', 'median'),
        maximum_origin_wape=('pooled_wape', 'max'),
    )
    .reset_index()
)
origin_stability['grain'] = pd.Categorical(
    origin_stability.grain, categories=GRAIN_ORDER, ordered=True
)
origin_stability = origin_stability.sort_values('grain').reset_index(drop=True)
display(origin_stability[[
    'grain_label', 'origins', 'minimum_origin_wape',
    'median_origin_wape', 'maximum_origin_wape',
]].style.format({
    'origins': '{:,.0f}',
    'minimum_origin_wape': '{:.2%}',
    'median_origin_wape': '{:.2%}',
    'maximum_origin_wape': '{:.2%}',
}))

origin_wide = (
    origin_results.pivot(index='origin', columns='grain', values='pooled_wape')
    .reindex(columns=GRAIN_ORDER)
    .rename(columns=GRAIN_LABELS)
)
display(origin_wide.style.format('{:.2%}'))

,grain_label,origins,minimum_origin_wape,median_origin_wape,maximum_origin_wape
0,Article-store-day,20,49.06%,57.61%,68.42%
1,Article-store-week (7-day total),20,31.01%,36.36%,47.71%
2,Article-day (stores pooled),20,17.32%,23.38%,37.53%


grain,Article-store-day,Article-store-week (7-day total),Article-day (stores pooled)
origin,,,
2026-03-02 00:00:00,53.24%,35.15%,19.98%
2026-03-09 00:00:00,58.20%,35.90%,22.02%
2026-03-16 00:00:00,57.68%,35.99%,21.49%
2026-03-23 00:00:00,60.49%,37.16%,22.29%
2026-03-30 00:00:00,49.06%,34.09%,25.74%
2026-04-06 00:00:00,57.00%,37.32%,19.51%
2026-04-13 00:00:00,68.42%,47.71%,37.53%
2026-04-20 00:00:00,55.68%,36.18%,22.24%
2026-04-27 00:00:00,60.26%,41.08%,34.87%


In [5]:
result_by_grain = grain_summary.set_index('grain')
weekly_wape = result_by_grain.loc['article_store_week', 'pooled_wape']
article_day_wape = result_by_grain.loc[
    'article_day_across_stores', 'pooled_wape'
]
weekly_reduction = result_by_grain.loc[
    'article_store_week', 'wape_reduction_vs_daily_pp'
]
store_reduction = result_by_grain.loc[
    'article_day_across_stores', 'wape_reduction_vs_daily_pp'
]
ratio = result_by_grain.loc['article_store_day', 'ratio']

display(Markdown(f'''
## Interpretation

Weekly article-store WAPE is **{weekly_wape:.2%}**. This is far below the
daily result and reduces WAPE by **{weekly_reduction:.2f} percentage points**,
but it remains slightly above the proposed 25–35% noise-floor band. The result
therefore supports a mixed conclusion: daily arrival noise is a major source of
error, while some modelling headroom may remain at the article-store-week level.

Pooling stores at article-day level reaches **{article_day_wape:.2%}** WAPE, a
**{store_reduction:.2f}-point** reduction from the original daily grain and inside
the proposed noise-limited range. The unchanged forecast-to-actual ratio of
**{ratio:.3f}** shows that aggregation improves accuracy through cancellation of
store-day errors, not by correcting aggregate volume bias.

For reporting, Pseudo 890 is therefore much more defensible at a weekly
article-store grain or at an article-day portfolio grain than at the original
article-store-day grain.
'''))


## Interpretation

Weekly article-store WAPE is **37.37%**. This is far below the
daily result and reduces WAPE by **20.16 percentage points**,
but it remains slightly above the proposed 25–35% noise-floor band. The result
therefore supports a mixed conclusion: daily arrival noise is a major source of
error, while some modelling headroom may remain at the article-store-week level.

Pooling stores at article-day level reaches **25.72%** WAPE, a
**31.81-point** reduction from the original daily grain and inside
the proposed noise-limited range. The unchanged forecast-to-actual ratio of
**0.990** shows that aggregation improves accuracy through cancellation of
store-day errors, not by correcting aggregate volume bias.

For reporting, Pseudo 890 is therefore much more defensible at a weekly
article-store grain or at an article-day portfolio grain than at the original
article-store-day grain.


## Extension to the other sourcing groups

The identical no-retraining diagnostic is now applied to `FCM 890`, `FCM 900`, and `Pseudo 900`. Each group's daily WAPE is its own reference, and actual and forecast volume are again required to remain unchanged across grains.

In [6]:
OTHER_SEGMENT_ORDER = ['FCM 890', 'FCM 900', 'Pseudo 900']

con.execute(
    '''
    CREATE OR REPLACE TEMP TABLE other_groups_daily AS
    SELECT
        sourcing_group, category_id::INTEGER AS category_id,
        ARTIKEL_ID::BIGINT AS ARTIKEL_ID,
        MARKT_ID::BIGINT AS MARKT_ID,
        CAST(origin AS DATE) AS origin,
        CAST(period AS DATE) AS period,
        actual::DOUBLE AS actual,
        forecast::DOUBLE AS forecast
    FROM read_csv_auto(?)
    WHERE is_active
      AND NOT (sourcing_group = ? AND category_id = ?)
    ''',
    [str(forecast_path), SOURCE_GROUP, CATEGORY_ID],
)
found_segments = set(
    con.execute(
        """SELECT DISTINCT
            sourcing_group || ' ' || category_id::VARCHAR AS segment
        FROM other_groups_daily"""
    ).fetchnumpy()['segment']
)
if found_segments != set(OTHER_SEGMENT_ORDER):
    raise ValueError(f'Unexpected sourcing groups: {sorted(found_segments)}')

con.execute(
    '''
    CREATE OR REPLACE TEMP TABLE other_groups_weekly AS
    SELECT
        sourcing_group, category_id, ARTIKEL_ID, MARKT_ID, origin,
        SUM(actual) AS actual, SUM(forecast) AS forecast
    FROM other_groups_daily
    GROUP BY sourcing_group, category_id, ARTIKEL_ID, MARKT_ID, origin
    '''
)
con.execute(
    '''
    CREATE OR REPLACE TEMP TABLE other_groups_article_day AS
    SELECT
        sourcing_group, category_id, ARTIKEL_ID, origin, period,
        SUM(actual) AS actual, SUM(forecast) AS forecast
    FROM other_groups_daily
    GROUP BY sourcing_group, category_id, ARTIKEL_ID, origin, period
    '''
)

other_summary = con.execute(
    '''
    WITH grains AS (
        SELECT sourcing_group, category_id, 'article_store_day' AS grain,
               actual, forecast
        FROM other_groups_daily
        UNION ALL
        SELECT sourcing_group, category_id, 'article_store_week' AS grain,
               actual, forecast
        FROM other_groups_weekly
        UNION ALL
        SELECT sourcing_group, category_id,
               'article_day_across_stores' AS grain, actual, forecast
        FROM other_groups_article_day
    )
    SELECT
        sourcing_group || ' ' || category_id::VARCHAR AS segment,
        grain, COUNT(*) AS n_units,
        SUM(actual) AS actual_kg, SUM(forecast) AS forecast_kg,
        AVG(actual) AS mean_actual_kg_per_unit,
        SUM(ABS(forecast - actual)) AS absolute_error_kg,
        SUM(ABS(forecast - actual)) / NULLIF(SUM(actual), 0) AS pooled_wape,
        SUM(forecast) / NULLIF(SUM(actual), 0) AS ratio
    FROM grains
    GROUP BY sourcing_group, category_id, grain
    '''
).fetchdf()

for segment, segment_rows in other_summary.groupby('segment'):
    if not np.allclose(segment_rows.actual_kg, segment_rows.actual_kg.iloc[0]):
        raise ValueError(f'Actual kilograms changed across grains for {segment}')
    if not np.allclose(segment_rows.forecast_kg, segment_rows.forecast_kg.iloc[0]):
        raise ValueError(f'Forecast kilograms changed across grains for {segment}')

daily_reference = (
    other_summary.loc[
        other_summary.grain.eq('article_store_day'),
        ['segment', 'pooled_wape', 'absolute_error_kg'],
    ]
    .rename(columns={
        'pooled_wape': 'daily_wape',
        'absolute_error_kg': 'daily_absolute_error_kg',
    })
)
other_summary = other_summary.merge(
    daily_reference, on='segment', validate='many_to_one'
)
other_summary['wape_reduction_vs_daily_pp'] = 100 * (
    other_summary.daily_wape - other_summary.pooled_wape
)
other_summary['absolute_error_removed_vs_daily'] = (
    1 - other_summary.absolute_error_kg / other_summary.daily_absolute_error_kg
)
other_summary['pp_vs_zero'] = 100 * (1 - other_summary.pooled_wape)
other_summary['segment'] = pd.Categorical(
    other_summary.segment, categories=OTHER_SEGMENT_ORDER, ordered=True
)
other_summary['grain'] = pd.Categorical(
    other_summary.grain, categories=GRAIN_ORDER, ordered=True
)
other_summary = other_summary.sort_values(
    ['segment', 'grain']
).reset_index(drop=True)
other_summary['grain_label'] = other_summary.grain.map(GRAIN_LABELS)

display(other_summary[[
    'segment', 'grain_label', 'n_units', 'mean_actual_kg_per_unit',
    'actual_kg', 'forecast_kg', 'pooled_wape', 'ratio',
    'wape_reduction_vs_daily_pp', 'absolute_error_removed_vs_daily',
    'pp_vs_zero',
]].style.format({
    'n_units': '{:,.0f}', 'mean_actual_kg_per_unit': '{:,.3f}',
    'actual_kg': '{:,.1f}', 'forecast_kg': '{:,.1f}',
    'pooled_wape': '{:.2%}', 'ratio': '{:.3f}',
    'wape_reduction_vs_daily_pp': '{:+.2f}',
    'absolute_error_removed_vs_daily': '{:.1%}',
    'pp_vs_zero': '{:+.2f}',
}))

,segment,grain_label,n_units,mean_actual_kg_per_unit,actual_kg,forecast_kg,pooled_wape,ratio,wape_reduction_vs_daily_pp,absolute_error_removed_vs_daily,pp_vs_zero
0,FCM 890,Article-store-day,"33,060",0.116,"3,840.1","5,993.3",181.72%,1.561,+0.00,0.0%,-81.72
1,FCM 890,Article-store-week (7-day total),"5,727",0.671,"3,840.1","5,993.3",111.04%,1.561,+70.68,38.9%,-11.04
2,FCM 890,Article-day (stores pooled),750,5.120,"3,840.1","5,993.3",90.23%,1.561,+91.49,50.3%,+9.77
3,FCM 900,Article-store-day,"315,810",0.180,"56,737.4","59,014.3",84.29%,1.040,+0.00,0.0%,+15.71
4,FCM 900,Article-store-week (7-day total),"54,827",1.035,"56,737.4","59,014.3",48.61%,1.040,+35.67,42.3%,+51.39
5,FCM 900,Article-day (stores pooled),"2,410",23.542,"56,737.4","59,014.3",23.05%,1.040,+61.23,72.6%,+76.95
6,Pseudo 900,Article-store-day,"20,162",0.145,"2,921.6","3,306.2",102.39%,1.132,+0.00,0.0%,-2.39
7,Pseudo 900,Article-store-week (7-day total),"3,508",0.833,"2,921.6","3,306.2",67.24%,1.132,+35.15,34.3%,+32.76
8,Pseudo 900,Article-day (stores pooled),"2,039",1.433,"2,921.6","3,306.2",41.27%,1.132,+61.12,59.7%,+58.73


In [7]:
other_origin_results = con.execute(
    '''
    WITH grains AS (
        SELECT sourcing_group, category_id, 'article_store_day' AS grain,
               origin, actual, forecast
        FROM other_groups_daily
        UNION ALL
        SELECT sourcing_group, category_id, 'article_store_week' AS grain,
               origin, actual, forecast
        FROM other_groups_weekly
        UNION ALL
        SELECT sourcing_group, category_id,
               'article_day_across_stores' AS grain, origin, actual, forecast
        FROM other_groups_article_day
    )
    SELECT
        sourcing_group || ' ' || category_id::VARCHAR AS segment,
        grain, origin,
        SUM(ABS(forecast - actual)) / NULLIF(SUM(actual), 0) AS pooled_wape
    FROM grains
    GROUP BY sourcing_group, category_id, grain, origin
    '''
).fetchdf()
other_origin_stability = (
    other_origin_results.groupby(['segment', 'grain'], observed=True)
    .agg(
        origins=('origin', 'nunique'),
        minimum_origin_wape=('pooled_wape', 'min'),
        median_origin_wape=('pooled_wape', 'median'),
        maximum_origin_wape=('pooled_wape', 'max'),
    )
    .reset_index()
)
other_origin_stability['segment'] = pd.Categorical(
    other_origin_stability.segment, categories=OTHER_SEGMENT_ORDER, ordered=True
)
other_origin_stability['grain'] = pd.Categorical(
    other_origin_stability.grain, categories=GRAIN_ORDER, ordered=True
)
other_origin_stability = other_origin_stability.sort_values(
    ['segment', 'grain']
).reset_index(drop=True)
other_origin_stability['grain_label'] = (
    other_origin_stability.grain.map(GRAIN_LABELS)
)
display(other_origin_stability[[
    'segment', 'grain_label', 'origins', 'minimum_origin_wape',
    'median_origin_wape', 'maximum_origin_wape',
]].style.format({
    'origins': '{:,.0f}', 'minimum_origin_wape': '{:.2%}',
    'median_origin_wape': '{:.2%}', 'maximum_origin_wape': '{:.2%}',
}))

def other_value(segment, grain, column):
    return other_summary.loc[
        other_summary.segment.eq(segment) & other_summary.grain.eq(grain), column
    ].iloc[0]

fcm_890_weekly = other_value('FCM 890', 'article_store_week', 'pooled_wape')
fcm_890_stores = other_value(
    'FCM 890', 'article_day_across_stores', 'pooled_wape'
)
fcm_890_ratio = other_value('FCM 890', 'article_store_day', 'ratio')
fcm_900_weekly = other_value('FCM 900', 'article_store_week', 'pooled_wape')
fcm_900_stores = other_value(
    'FCM 900', 'article_day_across_stores', 'pooled_wape'
)
fcm_900_ratio = other_value('FCM 900', 'article_store_day', 'ratio')
pseudo_900_weekly = other_value(
    'Pseudo 900', 'article_store_week', 'pooled_wape'
)
pseudo_900_stores = other_value(
    'Pseudo 900', 'article_day_across_stores', 'pooled_wape'
)
pseudo_900_ratio = other_value('Pseudo 900', 'article_store_day', 'ratio')

display(Markdown(f'''
### Interpretation of the other groups

- **FCM 890:** Weekly WAPE remains **{fcm_890_weekly:.2%}** and pooling
  stores only lowers it to **{fcm_890_stores:.2%}**. The ratio of
  **{fcm_890_ratio:.3f}** exposes severe aggregate overforecasting. This is not
  merely daily arrival noise; the forecast level has substantial headroom.
- **FCM 900:** Weekly WAPE is **{fcm_900_weekly:.2%}**, while the
  across-store article-day result reaches **{fcm_900_stores:.2%}** with a ratio
  of **{fcm_900_ratio:.3f}**. Store-level noise is important, but the weekly
  article-store grain still leaves modelling headroom.
- **Pseudo 900:** Weekly WAPE remains **{pseudo_900_weekly:.2%}** and the
  across-store result remains **{pseudo_900_stores:.2%}**. Together with the
  **{pseudo_900_ratio:.3f}** ratio, this indicates remaining level and calibration
  error rather than a purely noise-limited daily problem.
'''))

,segment,grain_label,origins,minimum_origin_wape,median_origin_wape,maximum_origin_wape
0,FCM 890,Article-store-day,20,82.25%,314.62%,3498.32%
1,FCM 890,Article-store-week (7-day total),20,44.53%,253.61%,3438.66%
2,FCM 890,Article-day (stores pooled),20,28.23%,260.86%,3455.12%
3,FCM 900,Article-store-day,20,69.69%,85.96%,97.49%
4,FCM 900,Article-store-week (7-day total),20,39.02%,49.86%,59.03%
5,FCM 900,Article-day (stores pooled),20,9.81%,22.70%,38.14%
6,Pseudo 900,Article-store-day,20,69.58%,99.36%,178.76%
7,Pseudo 900,Article-store-week (7-day total),20,37.74%,66.11%,140.67%
8,Pseudo 900,Article-day (stores pooled),20,17.37%,41.17%,127.88%



### Interpretation of the other groups

- **FCM 890:** Weekly WAPE remains **111.04%** and pooling
  stores only lowers it to **90.23%**. The ratio of
  **1.561** exposes severe aggregate overforecasting. This is not
  merely daily arrival noise; the forecast level has substantial headroom.
- **FCM 900:** Weekly WAPE is **48.61%**, while the
  across-store article-day result reaches **23.05%** with a ratio
  of **1.040**. Store-level noise is important, but the weekly
  article-store grain still leaves modelling headroom.
- **Pseudo 900:** Weekly WAPE remains **67.24%** and the
  across-store result remains **41.27%**. Together with the
  **1.132** ratio, this indicates remaining level and calibration
  error rather than a purely noise-limited daily problem.


## Scope of the finding

This is a post-processing diagnostic, not an estimate of a formal irreducible-error lower bound. It demonstrates how much error cancels when the *existing daily forecasts* are reported at coarser grains. It does not test whether a model trained directly on weekly targets could outperform aggregated daily forecasts. The conclusion is therefore about suitable evaluation and reporting granularity, with the remaining weekly error still leaving room for targeted model improvements. Per-origin WAPE is denominator-sensitive for very low-volume groups—especially FCM 890—so pooled WAPE remains the primary group-level measure and the origin ranges are stability diagnostics.